In [1]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu pypdf gradio groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [6]:
import requests

pdf_url = "https://drive.google.com/uc?export=download&id=1WdjXAzel69SlTiULSm2QO6v6ufrcktgP"   # <-- replace this
pdf_path = "document.pdf"

response = requests.get(pdf_url)

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF downloaded!")

PDF downloaded!


In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages")

Loaded 395 pages


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Total chunks: {len(docs)}")

Total chunks: 790


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [13]:
from google.colab import userdata
from groq import Groq

# Fetch API key securely
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

In [24]:
def rag_answer(query):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
You are a helpful assistant.

Answer ONLY using the context below.
If not found, say "I don't know".

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",   # ✅ UPDATED
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response.choices[0].message.content

    return answer

In [26]:
import gradio as gr

# Store chat history
def chat_fn(message, history):
    if history is None:
        history = []

    response = rag_answer(message)

    history.append((message, response))
    return history, history


with gr.Blocks(theme=gr.themes.Soft(), css="""
#chatbox {height: 500px; overflow-y: auto;}
""") as demo:

    gr.Markdown("""
    # 📄 AI PDF Assistant
    Ask questions from your documents
    """)

    chatbot = gr.Chatbot(elem_id="chatbox")

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type your question here...",
            scale=8,
            show_label=False
        )
        send_btn = gr.Button("Send", scale=1)

    clear_btn = gr.Button("🧹 Clear Chat")

    state = gr.State([])

    # Send message
    send_btn.click(
        chat_fn,
        inputs=[msg, state],
        outputs=[chatbot, state]
    )

    # Press Enter to send
    msg.submit(
        chat_fn,
        inputs=[msg, state],
        outputs=[chatbot, state]
    )

    # Clear chat
    clear_btn.click(
        lambda: ([], []),
        None,
        [chatbot, state]
    )

demo.launch(share=True)

/tmp/ipykernel_5439/298646355.py:14: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css="""
/tmp/ipykernel_5439/298646355.py:14: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css="""
/tmp/ipykernel_5439/298646355.py:23: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(elem_id="chatbox")
/tmp/ipykernel_5439/298646355.py:23: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bad5a3cc10283e7824.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
